# CarRacing-v3 — Gymnasium Step-by-Step Exploration

**Docs:** https://gymnasium.farama.org/environments/box2d/car_racing/#starting-state

This notebook walks through the CarRacing environment piece by piece — from loading it all the way to watching it run as an animation.

---

### What is CarRacing?

CarRacing-v3 is a continuous-control environment where a top-down race car must drive along a randomly generated track.

| Property | Value |
|---|---|
| **Observation** | 96×96 RGB image (what the agent "sees") |
| **Action space** | Continuous — `[steering, gas, brake]` |
| **Reward** | +1000/N per new track tile visited, −0.1 per frame |
| **Episode end** | All tiles visited, or car goes off track for too long |

The challenge for a DRL agent: learn to steer, accelerate, and brake from **pixels alone**.

---
## Step 1 — Imports

We need:
- `gymnasium` — the environment itself
- `numpy` — numerical operations on observations (pixel arrays)
- `matplotlib` — rendering frames as images
- `FuncAnimation` + `HTML` — stitching frames into an inline video

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

---
## Step 2 — Create the environment

`gym.make()` initialises the environment. The key argument here is `render_mode`:

| render_mode | Effect |
|---|---|
| `"human"` | Opens a live GUI window (won't work in Jupyter) |
| `"rgb_array"` | Returns each frame as a NumPy array — what we want |

After creating it, we call `env.reset()` to get the **initial observation** and start a fresh episode.
The `seed` makes the random track reproducible.

In [ ]:
env = gym.make("CarRacing-v3", render_mode="rgb_array")
obs, info = env.reset(seed=42)

print("Environment created and reset.")
print()
print("Observation space:", env.observation_space)
# Box(0, 255, (96, 96, 3), uint8)  →  96×96 pixel image with 3 colour channels

print("Action space:     ", env.action_space)
# Box([-1.  0.  0.], [1. 1. 1.], (3,), float32)
# Index 0 = steering  (-1 full left  →  +1 full right)
# Index 1 = gas       ( 0 = off     →  +1 full throttle)
# Index 2 = brake     ( 0 = off     →  +1 full brake)

---
## Step 3 — Inspect the observation

The observation returned by `reset()` is the first frame the agent sees — a 96×96 RGB image stored as a NumPy array.

Understanding its shape and range is important before you build a neural network on top of it.

In [ ]:
print("obs.shape :", obs.shape)   # (96, 96, 3) — height × width × channels
print("obs.dtype :", obs.dtype)   # uint8  — pixel values 0–255
print("obs.min() :", obs.min())   # 0
print("obs.max() :", obs.max())   # 255
print()
print("Top-left pixel (R, G, B):", obs[0, 0])   # one sample pixel

---
## Step 4 — Render and display a single frame

`env.render()` returns the current frame as a NumPy array when `render_mode="rgb_array"`.
We pass it straight to `plt.imshow()` to display it.

At reset the car is placed at the start of the track, zoomed out to show the full circuit before zooming in.

In [ ]:
frame = env.render()   # returns a (96, 96, 3) uint8 array

plt.figure(figsize=(5, 5))
plt.imshow(frame)
plt.axis("off")
plt.title("CarRacing — frame at reset (seed 42)")
plt.show()

print("Rendered frame shape:", frame.shape)

---
## Step 5 — Take one action

The core loop of RL is:

```
action  →  env.step(action)  →  (next_obs, reward, terminated, truncated, info)
```

Here we sample a **random** action from the action space to see what `step()` returns.

| Return value | Meaning |
|---|---|
| `obs` | Next observation (96×96 image after the action) |
| `reward` | Score change this step |
| `terminated` | Episode ended naturally (e.g. finished lap) |
| `truncated` | Episode cut off by a time/step limit |
| `info` | Extra diagnostic data (usually empty here) |

In [ ]:
action = env.action_space.sample()   # random action from valid range

print("Random action sampled:")
print(f"  steering = {action[0]: .3f}  (−1=left, +1=right)")
print(f"  gas      = {action[1]: .3f}  ( 0=off,  +1=full)")
print(f"  brake    = {action[2]: .3f}  ( 0=off,  +1=full)")
print()

obs, reward, terminated, truncated, info = env.step(action)

print("After stepping:")
print(f"  reward     = {reward:.4f}")
print(f"  terminated = {terminated}")
print(f"  truncated  = {truncated}")
print(f"  info       = {info}")

---
## Step 6 — Side-by-side: RGB vs Grayscale

Most DRL implementations convert the 3-channel RGB observation to grayscale before feeding it to a CNN. This:
- Reduces input size by 3×
- Removes colour information the agent doesn't really need

The standard luminance formula is: `Y = 0.299R + 0.587G + 0.114B`

In [ ]:
# Reset to get a clean starting frame
env2 = gym.make("CarRacing-v3", render_mode="rgb_array")
obs2, _ = env2.reset(seed=42)
env2.close()

# Convert to grayscale using luminance weights
gray = np.dot(obs2[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(obs2)
axes[0].set_title(f"RGB observation  {obs2.shape}")
axes[0].axis("off")

axes[1].imshow(gray, cmap="gray")
axes[1].set_title(f"Grayscale  {gray.shape}  (DRL pre-processing)")
axes[1].axis("off")

plt.suptitle("What the agent sees", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 7 — Run a full episode and animate it

Now we put the whole loop together:

1. Reset the environment
2. Collect frames while stepping with random actions
3. Stop when `terminated` or `truncated` (or we hit our frame cap)
4. Stitch the frames into an inline animation with `FuncAnimation`

A random policy will drive chaotically — but this lets us verify the environment works end-to-end before we add a real agent.

In [ ]:
env3 = gym.make("CarRacing-v3", render_mode="rgb_array")
obs3, info = env3.reset(seed=42)

frames       = []
rewards      = []
terminated   = False
truncated    = False
MAX_FRAMES   = 300   # cap so the cell doesn't run forever

# ── Main loop ──────────────────────────────────────────────────
while not (terminated or truncated) and len(frames) < MAX_FRAMES:
    frames.append(env3.render())              # capture frame BEFORE the step
    action = env3.action_space.sample()       # random policy
    obs3, reward, terminated, truncated, _ = env3.step(action)
    rewards.append(reward)
# ───────────────────────────────────────────────────────────────

env3.close()

print(f"Frames collected : {len(frames)}")
print(f"Total reward     : {sum(rewards):.2f}")
print(f"Mean reward/step : {np.mean(rewards):.4f}")
print(f"Episode ended    : terminated={terminated}  truncated={truncated}")

In [ ]:
# Build the animation from collected frames
fig, ax = plt.subplots(figsize=(5, 5))
ax.axis("off")
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return [img]

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=30,   # ms between frames ≈ 33 fps
    blit=True
)
plt.close(fig)   # prevent a static extra image appearing

HTML(ani.to_jshtml())   # renders as an interactive player in the notebook

---
## Step 8 — Reward curve

Plot the per-step reward over the episode. You should see:
- Negative reward (−0.1/frame) during the first ~50 frames while the camera zooms in
- Occasional positive spikes when the car crosses a new track tile
- Mostly negative with a random policy — the car usually crashes or spins

In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(rewards, color="steelblue", linewidth=1)
plt.axhline(0, color="red", linewidth=0.8, linestyle="--", label="zero")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.title(f"Per-step reward — random policy ({len(rewards)} steps, total={sum(rewards):.1f})")
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary — what we learned

| Concept | Detail |
|---|---|
| `gym.make("CarRacing-v3", render_mode="rgb_array")` | Creates the environment in headless mode |
| `env.reset(seed=42)` | Starts a new episode, returns first observation |
| `obs.shape` | `(96, 96, 3)` — a 96×96 RGB image |
| `env.action_space` | Continuous Box `[steering, gas, brake]` |
| `env.step(action)` | Advances the sim one step, returns obs/reward/done flags |
| `env.render()` | Returns the current frame as a NumPy array |
| Grayscale pre-processing | `np.dot(obs[...,:3], [0.299, 0.587, 0.114])` |

**Next step:** replace `env.action_space.sample()` with a trained policy (e.g. PPO or DQN).